
# Step-by-Step ChromaDB Tutorial (Production-Oriented)

This notebook provides a **step-by-step introduction to ChromaDB**, a popular open‑source vector database used in Retrieval Augmented Generation (RAG) systems.

The tutorial gradually moves from **basic concepts to advanced usage**, including:

- Creating and managing vector collections
- Storing embeddings and documents
- Performing semantic search
- Using metadata filters
- Custom embedding models
- Updating and deleting documents
- Building a simple RAG retrieval pipeline

Notebook generated on: **2026-03-12**



## Step 1 — Install ChromaDB

Create a fresh Python environment and install required dependencies.

Required packages:

- chromadb
- sentence-transformers
- pandas (optional for data manipulation)
- langchain (optional for integration with LLM pipelines)

Run the following commands in your terminal before running the notebook.


In [ ]:

# Run these in a terminal if not already installed:
# pip install chromadb
# pip install sentence-transformers
# pip install pandas
# pip install langchain



## Step 2 — Initialize ChromaDB

ChromaDB supports two main modes:

1. **In-memory database** (good for experiments)
2. **Persistent database** (recommended for real applications)

Below we initialize a simple in-memory client.


In [1]:

import chromadb

client = chromadb.Client()

print("ChromaDB client initialized")


ChromaDB client initialized



## Step 3 — Create a Collection

A **collection** in ChromaDB is similar to a table in a relational database.

Each collection stores:

- documents
- embeddings
- metadata
- ids

We create a collection for company policy documents.


In [2]:

collection = client.create_collection(name="company_policies")

print("Collection created")


Collection created



## Step 4 — Insert Documents

Let’s simulate a small **enterprise policy dataset**.

These documents could represent:

- HR policies
- security policies
- remote work rules


In [3]:

documents = [
    "Employees are entitled to 20 days of annual leave per year.",
    "Travel reimbursement requires manager approval.",
    "Passwords must be changed every 90 days.",
    "Employees can work remotely two days per week.",
    "Maternity leave is available for 26 weeks."
]

ids = ["doc1", "doc2", "doc3", "doc4", "doc5"]

collection.add(
    documents=documents,
    ids=ids
)

print("Documents inserted into ChromaDB")


Documents inserted into ChromaDB



## Step 5 — Perform Semantic Search

One of the key advantages of vector databases is **semantic similarity search**.

Instead of matching keywords, the system retrieves documents based on **meaning**.


In [4]:

results = collection.query(
    query_texts=["How many leave days do employees get?"],
    n_results=2
)

results["documents"]


[['Employees are entitled to 20 days of annual leave per year.',
  'Employees can work remotely two days per week.']]


## Step 6 — Use Metadata

Metadata allows us to attach structured information to each document.

Example metadata:

- department
- document type
- year


In [5]:

collection.add(
    documents=[
        "Employees get 20 days of annual leave",
        "Travel expenses require approval",
        "Password must change every 90 days"
    ],
    ids=["m1", "m2", "m3"],
    metadatas=[
        {"department": "HR"},
        {"department": "Finance"},
        {"department": "IT"}
    ]
)

print("Documents with metadata inserted")


Documents with metadata inserted



## Step 7 — Query with Metadata Filtering

Metadata filtering helps restrict search results to **specific categories**.


In [6]:

results = collection.query(
    query_texts=["leave policy"],
    where={"department": "HR"}  # Experiment without this line and justify the results
)

results


{'ids': [['m1']],
 'embeddings': None,
 'documents': [['Employees get 20 days of annual leave']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'department': 'HR'}]],
 'distances': [[1.0194077491760254]]}


## Step 8 — Use Custom Embedding Models

Instead of default embeddings, we can use models from **Sentence Transformers**.


In [ ]:
!pip3 install sentence-transformers

In [ ]:
import sys
sys.executable

In [7]:
from sentence_transformers import SentenceTransformer

In [8]:
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded



## Step 9 — Create Collection with Custom Embeddings

We define a function that converts text into embeddings.


In [ ]:

'''
# Older method creates errors
custom_collection = client.create_collection(
    name="policy_docs",
    embedding_function=embed
)
'''
'''
custom_collection = client.get_or_create_collection(
    name="policy_docs",
    embedding_function=embed
)

'''


In [9]:
class MyEmbeddingFunction:
    def __init__(self, model):
        self.model = model

    def __call__(self, input): # __add__
        # Used for documents
        return self.model.encode(input).tolist()

    # important for queries
    def embed_query(self, input):
        # Used for queries
        return self.model.encode(input).tolist()


# emb_fun = lambda input : model.encode(input).tolist()

embedding_fn = MyEmbeddingFunction(model)

# client.delete_collection(name="policy_docs")
custom_collection = client.create_collection(
    name="policy_docs",
    embedding_function=embedding_fn
)


## Step 10 — Batch Insert Documents

For large datasets we typically insert documents in batches.


In [10]:

documents_batch = [
    "Annual leave policy for employees.",
    "Expense reimbursement guidelines.",
    "Security and password management rules.",
    "Remote work guidelines."
]

ids_batch = [f"id{i}" for i in range(len(documents_batch))]

custom_collection.add(
    documents=documents_batch,
    ids=ids_batch
)

print("Batch documents inserted")


Batch documents inserted



## Step 11 — Store Embeddings Directly

Sometimes embeddings are generated externally (for example using OpenAI or other APIs).


In [11]:

embeddings = model.encode(documents_batch).tolist()
print("Embeddings: ", embeddings)


Embeddings:  [[0.03157556802034378, 0.06579717993736267, 0.03483077138662338, 0.021718226373195648, 0.07771220058202744, 0.1013716459274292, 0.007153687532991171, -0.10173306614160538, -0.04853945970535278, -0.023728765547275543, 0.051852814853191376, 0.027457933872938156, -0.07345771044492722, -0.00040917753358371556, 0.012610388919711113, 0.009944909252226353, -0.0053214989602565765, 0.008577435277402401, 0.01283392496407032, -0.07869309931993484, -0.007211841177195311, -0.050749652087688446, -0.0446290522813797, 0.03433765470981598, -0.019533704966306686, 0.02587764710187912, 0.0031069391407072544, 0.028237229213118553, -0.04111857712268829, 0.07553993910551071, 0.018025485798716545, 0.01059585902839899, -0.01701413094997406, -0.049676407128572464, -0.01767548732459545, 0.02166120335459709, -0.0452861525118351, -0.01774383708834648, -0.02566653862595558, 0.02808375656604767, -0.047830626368522644, 0.011018318124115467, 0.03815716877579689, 0.008658114820718765, 0.0008044002461247146

In [12]:
custom_collection.add(
    embeddings=embeddings,
    documents=documents_batch,
    ids=[f"emb{i}" for i in range(len(documents_batch))]
)

print("Documents with external embeddings stored")

Documents with external embeddings stored



## Step 12 — Update and Delete Documents

ChromaDB supports updating or deleting documents.


In [13]:

# Update
custom_collection.update(
    ids=["id0"],
    documents=["Employees get 25 days of annual leave per year"]
)

# Delete
custom_collection.delete(ids=["id1"])

print("Update and delete operations completed")


Update and delete operations completed



## Step 13 — Inspect Collection

Useful commands to inspect collections.


In [14]:

print("Document count:", custom_collection.count())

print("Peek documents:")
custom_collection.peek()


Document count: 7
Peek documents:


{'ids': ['id0', 'id2', 'id3', 'emb0', 'emb1', 'emb2', 'emb3'],
 'embeddings': array([[ 0.07038206,  0.01818221,  0.05550462, ..., -0.02345401,
          0.04196787,  0.01994185],
        [-0.02668818,  0.01933625, -0.08029221, ...,  0.04198882,
          0.02214986, -0.026489  ],
        [-0.0287434 ,  0.00097479,  0.02530241, ...,  0.06525033,
         -0.11450797,  0.02154844],
        ...,
        [-0.03313633,  0.07145616, -0.00087073, ..., -0.02557477,
          0.01851057, -0.07042726],
        [-0.02668818,  0.01933625, -0.08029221, ...,  0.04198882,
          0.02214986, -0.026489  ],
        [-0.0287434 ,  0.00097479,  0.02530241, ...,  0.06525033,
         -0.11450797,  0.02154844]], shape=(7, 384)),
 'documents': ['Employees get 25 days of annual leave per year',
  'Security and password management rules.',
  'Remote work guidelines.',
  'Annual leave policy for employees.',
  'Expense reimbursement guidelines.',
  'Security and password management rules.',
  'Remote work gu


## Step 14 — Build a Simple Retrieval Pipeline

In a RAG system, retrieval works as:

1. User query
2. Embedding generation
3. Vector search
4. Return top documents


In [15]:

query = "What is the leave policy?"

results = custom_collection.query(
    query_texts=[query],
    n_results=3
)

context = results["documents"]

context


[['Annual leave policy for employees.',
  'Employees get 25 days of annual leave per year',
  'Remote work guidelines.']]


## Step 15 — Persistent Storage

For production systems, the vector database must persist on disk.


In [16]:

import chromadb

persistent_client = chromadb.PersistentClient(path="./chroma_db")

collection = persistent_client.get_or_create_collection("persistent_docs")

collection.add(
    documents=["Example persistent document"],
    ids=["p1"]
)

print("✅ Database persisted automatically")


✅ Database persisted automatically
